# Notebook 2 Explained: Classification and Tuning Guide

This notebook explains every code snippet from Notebook 2.

For each snippet, you get:
- Why it is used
- How it works
- How to change it if needed
- A matching code snippet you can run

## Snippet 1 (Notebook 2 - Cell 3): Dependency Check and Installation

### Why this is used
It ensures all required libraries are available before model building starts.

### How it works
- Defines a list of required packages.
- Tries importing each package.
- Installs missing ones with pip using the active Python executable.

### How to change it
- Add/remove package names in the list.
- If your environment is stable, replace this cell with only import statements.
- Use requirements.txt in production workflows instead of notebook installation.

In [ ]:
# Snippet 1: Dependency check
import subprocess
import sys

packages = ['pandas', 'numpy', 'matplotlib', 'seaborn', 'scikit-learn', 'joblib']
for package in packages:
    try:
        __import__(package)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print('Dependencies ready')

## Snippet 2 (Notebook 2 - Cell 4): Imports and Global Settings

### Why this is used
It loads all libraries needed for data prep, classification, evaluation, and plotting.

### How it works
- Imports pandas/numpy/matplotlib/seaborn/joblib.
- Imports scikit-learn components for preprocessing, models, and metrics.
- Sets warning behavior, random seed, and plotting/display defaults.

### How to change it
- Remove unused imports for cleaner code.
- Change random seed if you want a different reproducible split.
- Adjust figure size and display options for report style.

In [ ]:
# Snippet 2: Imports and settings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Snippet 3 (Notebook 2 - Cell 5): Load Processed Artifacts from Notebook 1

### Why this is used
It loads prepared data and encoders from Notebook 1, which prevents repeating preprocessing.

### How it works
- Uses pathlib to build robust candidate paths.
- Searches current folder and parent folder for required joblib files.
- Raises clear FileNotFoundError if files are missing.
- Loads dataframe and encoder dictionary with joblib.load.

### How to change it
- Add more search folders if your artifacts are stored elsewhere.
- Replace hardcoded names with config variables.
- Use try/except if you want soft failure with a warning.

In [ ]:
# Snippet 3: Load artifacts
from pathlib import Path

base_dir = Path.cwd()
candidates = [base_dir, base_dir.parent]

data_path = next((p / 'data_processed.joblib' for p in candidates if (p / 'data_processed.joblib').exists()), None)
encoder_path = next((p / 'label_encoders.joblib' for p in candidates if (p / 'label_encoders.joblib').exists()), None)

if data_path is None or encoder_path is None:
    searched = ', '.join(str(p) for p in candidates)
    raise FileNotFoundError(f'Missing artifacts. Searched: {searched}')

df_processed = joblib.load(data_path)
label_encoders = joblib.load(encoder_path)
print(df_processed.shape)

## Snippet 4 (Notebook 2 - Cell 7): Algorithm Information Table

### Why this is used
It documents model families, learnable parameters, and key hyperparameters for report discussion.

### How it works
- Creates a list of dictionaries for NB, LR, and KNN.
- Converts the list into a pandas dataframe.
- Prints formatted table text.

### How to change it
- Add more algorithms if you expand experiments.
- Add extra columns like complexity or training time.
- Export the table to CSV for coursework appendix.

In [ ]:
# Snippet 4: Algorithm info table
algorithm_info = pd.DataFrame([
    {'Algorithm': 'Naive Bayes (NB)', 'Type': 'Non-parametric (Probabilistic)', 'Learnable Parameters': 'Class priors, conditionals', 'Strategic Hyperparameters': 'var_smoothing'},
    {'Algorithm': 'Logistic Regression (LR)', 'Type': 'Parametric (Statistical)', 'Learnable Parameters': 'Weights, bias', 'Strategic Hyperparameters': 'C, penalty'},
    {'Algorithm': 'K-Nearest Neighbour (KNN)', 'Type': 'Non-parametric (Instance-based)', 'Learnable Parameters': 'None (lazy)', 'Strategic Hyperparameters': 'n_neighbors, metric'}
])
print(algorithm_info.to_string(index=False))

## Snippet 5 (Notebook 2 - Cell 8): Feature/Target Preparation

### Why this is used
It defines input features and target for the classification task.

### How it works
- Drops identifier, target, and selected original categorical columns from features.
- Sets target series from loan_approval_status.

### How to change it
- Update drop list if you engineer new features.
- Keep original categorical columns only if you encode them in pipeline.
- Validate no leakage features are left in X_approval.

In [ ]:
# Snippet 5: Prepare X and y
X_approval = df_processed.drop(['id', 'loan_approval_status', 'max_allowed_loan', 'home_ownership', 'loan_intent', 'payment_default_on_file'], axis=1)
y_approval = df_processed['loan_approval_status']
print(X_approval.shape, y_approval.shape)

## Snippet 6 (Notebook 2 - Cell 9): Train/Test Split, Imputation, and Scaling

### Why this is used
It prepares clean and normalized train/test inputs for fair model training.

### How it works
- Splits data with stratify to preserve class balance.
- Imputes missing values with mean based on training data only.
- Scales features using StandardScaler fitted on training set.

### How to change it
- Change test_size for different evaluation setup.
- Use median imputation for skewed variables.
- Use RobustScaler if strong outliers hurt standard scaling.

In [ ]:
# Snippet 6: Split, impute, scale
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_approval, y_approval, test_size=0.2, random_state=42, stratify=y_approval
)

imputer = SimpleImputer(strategy='mean')
X_train_clf_imputed = pd.DataFrame(imputer.fit_transform(X_train_clf), columns=X_train_clf.columns, index=X_train_clf.index)
X_test_clf_imputed = pd.DataFrame(imputer.transform(X_test_clf), columns=X_test_clf.columns, index=X_test_clf.index)

scaler_clf = StandardScaler()
X_train_clf_scaled = scaler_clf.fit_transform(X_train_clf_imputed)
X_test_clf_scaled = scaler_clf.transform(X_test_clf_imputed)
print(X_train_clf_scaled.shape, X_test_clf_scaled.shape)

## Snippet 7 (Notebook 2 - Cell 10): Train Classification Models

### Why this is used
It trains three baseline models so performance can be compared objectively.

### How it works
- Initializes NB, LR, and KNN models.
- Fits each model on scaled training data.
- Stores models in a dictionary for later loop-based evaluation.

### How to change it
- Tune initial parameters (for example n_neighbors or C).
- Add new models like RandomForestClassifier.
- Use pipeline objects to keep preprocessing and model together.

In [ ]:
# Snippet 7: Train models
clf_models = {}

nb_clf = GaussianNB()
nb_clf.fit(X_train_clf_scaled, y_train_clf)
clf_models['Naive Bayes'] = nb_clf

lr_clf = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
lr_clf.fit(X_train_clf_scaled, y_train_clf)
clf_models['Logistic Regression'] = lr_clf

knn_clf = KNeighborsClassifier(n_neighbors=5)
knn_clf.fit(X_train_clf_scaled, y_train_clf)
clf_models['K-Nearest Neighbours'] = knn_clf

print(list(clf_models.keys()))

## Snippet 8 (Notebook 2 - Cell 12): Evaluate Models and Select Best

### Why this is used
It calculates key metrics, plots diagnostics, and identifies the best baseline model.

### How it works
- Loops over trained models.
- Computes predictions and probabilities.
- Calculates accuracy, precision, recall, F1, ROC-AUC.
- Plots confusion matrix and ROC curve.
- Stores metrics and picks model with highest F1-score.

### How to change it
- Change primary selection metric from F1 to ROC-AUC or Recall.
- Add cross-validation for more robust comparison.
- Save plots into a dedicated figures folder.

In [ ]:
# Snippet 8: Evaluate models (metric table version)
clf_results = []

for model_name, model in clf_models.items():
    y_pred = model.predict(X_test_clf_scaled)
    y_pred_proba = model.predict_proba(X_test_clf_scaled)[:, 1]

    clf_results.append({
        'Model': model_name,
        'Accuracy': accuracy_score(y_test_clf, y_pred),
        'Precision': precision_score(y_test_clf, y_pred, zero_division=0),
        'Recall': recall_score(y_test_clf, y_pred, zero_division=0),
        'F1-Score': f1_score(y_test_clf, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test_clf, y_pred_proba)
    })

clf_results_df = pd.DataFrame(clf_results)
best_clf_model = clf_results_df.loc[clf_results_df['F1-Score'].idxmax(), 'Model']
print(clf_results_df.to_string(index=False))
print('Best model:', best_clf_model)

## Snippet 9 (Notebook 2 - Cell 13): Metric Justification Table

### Why this is used
It explains which metrics should drive decisions for imbalanced classification.

### How it works
- Builds a manual decision table with USE/DO NOT USE labels.
- Prints selected metrics rationale next to test results.

### How to change it
- Add business-specific cost metrics (false rejection cost, false approval cost).
- Add PR-AUC if classes become highly imbalanced.
- Convert this table to markdown for report inclusion.

In [ ]:
# Snippet 9: Metrics decision table
metrics_table = pd.DataFrame([
    {'Metric': 'Accuracy', 'USE/DO NOT USE': 'DO NOT USE', 'Justification': 'Can mislead under imbalance'},
    {'Metric': 'Recall', 'USE/DO NOT USE': 'USE', 'Justification': 'Important for finding risky applicants'},
    {'Metric': 'Precision', 'USE/DO NOT USE': 'USE', 'Justification': 'Quality of predicted rejections'},
    {'Metric': 'F1-Score', 'USE/DO NOT USE': 'USE', 'Justification': 'Balances precision and recall'},
    {'Metric': 'AUC-ROC', 'USE/DO NOT USE': 'USE', 'Justification': 'Threshold-independent separation quality'}
])
print(metrics_table.to_string(index=False))
print(clf_results_df.to_string(index=False))

## Snippet 10 (Notebook 2 - Cell 14): Hyperparameter Tuning and Before/After Comparison

### Why this is used
It improves the best baseline model by searching a parameter grid and comparing improvement.

### How it works
- Chooses parameter grid based on the best baseline model type.
- Runs GridSearchCV with F1 scoring.
- Evaluates tuned model on test data.
- Compares before-vs-after metrics and confusion matrices.

### How to change it
- Expand parameter grid for deeper search.
- Use StratifiedKFold and repeated CV for stronger estimates.
- Switch scoring to recall if minimizing risky approvals is top priority.

In [ ]:
# Snippet 10: Hyperparameter tuning
if best_clf_model == 'Logistic Regression':
    param_grid = {'C': [0.01, 0.1, 1, 10, 100], 'penalty': ['l2'], 'solver': ['lbfgs']}
    base_clf = LogisticRegression(max_iter=1000, random_state=42)
elif best_clf_model == 'K-Nearest Neighbours':
    param_grid = {'n_neighbors': [3, 5, 7, 9, 11], 'metric': ['euclidean', 'manhattan']}
    base_clf = KNeighborsClassifier()
else:
    param_grid = {'var_smoothing': [1e-10, 1e-9, 1e-8, 1e-7, 1e-6]}
    base_clf = GaussianNB()

grid_search_clf = GridSearchCV(base_clf, param_grid, cv=5, scoring='f1', n_jobs=-1)
grid_search_clf.fit(X_train_clf_scaled, y_train_clf)

y_pred_tuned = grid_search_clf.predict(X_test_clf_scaled)
y_pred_proba_tuned = grid_search_clf.predict_proba(X_test_clf_scaled)[:, 1]

print('Best params:', grid_search_clf.best_params_)
print('Tuned F1:', f1_score(y_test_clf, y_pred_tuned))
print('Tuned ROC-AUC:', roc_auc_score(y_test_clf, y_pred_proba_tuned))

## Snippet 11 (Notebook 2 - Cell 16): Save Models for Reuse

### Why this is used
It saves trained artifacts so they can be reused in Notebook 3 or deployment without retraining.

### How it works
- Saves all baseline models dictionary.
- Saves best tuned estimator from GridSearchCV.
- Saves fitted scaler used for preprocessing.

### How to change it
- Save to a models folder for cleaner project structure.
- Add timestamps/version names to avoid accidental overwrite.
- Save metadata JSON (model name, score, training date) with artifacts.

In [ ]:
# Snippet 11: Save artifacts
joblib.dump(clf_models, 'clf_models.joblib')
joblib.dump(grid_search_clf.best_estimator_, 'best_clf_tuned.joblib')
joblib.dump(scaler_clf, 'scaler_clf.joblib')
print('Saved clf_models.joblib, best_clf_tuned.joblib, scaler_clf.joblib')

## Quick Change Templates

1. Change imputation method: SimpleImputer(strategy='median')
2. Change model selection criterion: idxmax on ROC-AUC instead of F1-Score
3. Change test size: train_test_split(..., test_size=0.25)
4. Add model: include RandomForestClassifier in clf_models
5. Save to folder: joblib.dump(model, 'models/model_name.joblib')